In [1]:
import pandas as pd
import numpy as np
import ast
import random
import numpy as np
from modules import DataFramePreprocessing
import torch
from torch.nn import Embedding
import mmh3
from transformers import get_linear_schedule_with_warmup,AutoTokenizer,DistilBertModel


In [2]:
interactions_train_df=pd.read_csv("data/interactions_train.csv")
PP_recipes_df=pd.read_csv("data/PP_recipes.csv")
PP_users_df=pd.read_csv("data/PP_users.csv")
RAW_interactions_df=pd.read_csv("data/RAW_interactions.csv")
RAW_recipes_df=pd.read_csv("data/RAW_recipes.csv")


In [3]:
interactions_train_df_filtered=interactions_train_df.sample(n=50000,replace=False)

In [4]:
print(interactions_train_df_filtered.shape)


(50000, 6)


In [5]:
RAW_recipes_df.rename(columns={"id":"recipe_id"},inplace=True)
PP_recipes_df.rename(columns={"techniques":"techniques_recipes"},inplace=True)
PP_users_df.rename(columns={"techniques":"techniques_users"},inplace=True)

In [6]:
print("interactions_train_df_filtered columns:")
print(interactions_train_df_filtered.columns)
print("------------------------------------------------------------------")
print("PP_recipes_df columns:")
print(PP_recipes_df.columns)
print("-------------------------------------------------------------------")
print("PP_users_df columns:")
print(PP_users_df.columns)
print("----------------------------------------------------------------------")
print("RAW_interactions_df columns:")
print(RAW_interactions_df.columns)
print("------------------------------------------------------------------------")
print("RAW_recipes_df columns:")
print(RAW_recipes_df.columns)

interactions_train_df_filtered columns:
Index(['user_id', 'recipe_id', 'date', 'rating', 'u', 'i'], dtype='object')
------------------------------------------------------------------
PP_recipes_df columns:
Index(['id', 'i', 'name_tokens', 'ingredient_tokens', 'steps_tokens',
       'techniques_recipes', 'calorie_level', 'ingredient_ids'],
      dtype='object')
-------------------------------------------------------------------
PP_users_df columns:
Index(['u', 'techniques_users', 'items', 'n_items', 'ratings', 'n_ratings'], dtype='object')
----------------------------------------------------------------------
RAW_interactions_df columns:
Index(['user_id', 'recipe_id', 'date', 'rating', 'review'], dtype='object')
------------------------------------------------------------------------
RAW_recipes_df columns:
Index(['name', 'recipe_id', 'minutes', 'contributor_id', 'submitted', 'tags',
       'nutrition', 'n_steps', 'steps', 'description', 'ingredients',
       'n_ingredients'],
      dty

In [7]:
full_df=interactions_train_df_filtered.merge(PP_recipes_df,on="i",how="left",suffixes=(None,"_1")).merge(PP_users_df,on="u",how="left",suffixes=(None,"_2")).merge(RAW_interactions_df,on=["user_id","recipe_id"],how="left",suffixes=(None,"_3")).merge(RAW_recipes_df,on=["recipe_id"],how="left",suffixes=(None,"_3"))

In [ ]:
full_df=full_df.drop(columns=["review",'rating_3',"id","ingredients","submitted","contributor_id","date_3","date","steps_tokens","ingredient_tokens","name_tokens","u"],axis=1)

In [9]:
print(f"Columns of the full dataframe:")
print(full_df.columns)
print("----------------------------------------")
print(f"Shape of the full dataframe:")
print(full_df.shape)
print("----------------------------------------")
print(f"Columns of the filtered interaction dataframe:")
print(interactions_train_df_filtered.shape)
print("----------------------------------------")
print(f"Head of the full  dataframe:")
display(full_df.head())
print("----------------------------------------")
for col in full_df.columns:
    print(f"Unique values of {col} :")
    print(full_df[col].unique())
    print("----------------------------------------")
for col in full_df.columns:
    print(f"The type of {col} is")
    print(full_df[col].apply(type).unique())
    print("---------------------------------")

Columns of the full dataframe:
Index(['user_id', 'recipe_id', 'rating', 'i', 'techniques_recipes',
       'calorie_level', 'ingredient_ids', 'techniques_users', 'items',
       'n_items', 'ratings', 'n_ratings', 'review', 'name', 'minutes', 'tags',
       'nutrition', 'n_steps', 'steps', 'description', 'n_ingredients'],
      dtype='object')
----------------------------------------
Shape of the full dataframe:
(50000, 21)
----------------------------------------
Columns of the filtered interaction dataframe:
(50000, 6)
----------------------------------------
Head of the full  dataframe:


,user_id,recipe_id,rating,i,techniques_recipes,calorie_level,ingredient_ids,techniques_users,items,n_items,...,n_ratings,review,name,minutes,tags,nutrition,n_steps,steps,description,n_ingredients
0,89366,151621,5.0,165827,"[0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, ...",2,"[5890, 1954, 5314, 840, 6335, 3203, 7705, 3810...","[4, 1, 0, 1, 2, 0, 1, 0, 0, 4, 0, 0, 0, 1, 0, ...","[109573, 68779, 176014, 14451, 165827, 69725, ...",8,...,8,Loved this! I made sure each of my shrimp were...,shrimp and pasta with creole cream sauce,48,"['60-minutes-or-less', 'time-to-make', 'course...","[906.9, 72.0, 2.0, 23.0, 96.0, 135.0, 24.0]",18,"['peel and devein shrimp', 'toss shrimp with c...",from southern living.,10
1,179133,275036,5.0,104393,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",1,"[2991, 3440, 5010, 3203, 840, 5006, 6273, 5180]","[357, 7, 3, 144, 158, 0, 3, 23, 2, 322, 13, 29...","[90890, 135961, 12856, 128256, 37047, 153154, ...",970,...,970,This was very nice! I used red bell pepper and...,easy weeknight corn,15,"['15-minutes-or-less', 'time-to-make', 'course...","[270.4, 20.0, 4.0, 3.0, 11.0, 37.0, 12.0]",6,"['in pan / dutch oven , add butter , olive oil...",i threw some things together in a dutch oven a...,8
2,1812036,32871,5.0,99923,"[1, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, ...",0,"[3729, 840, 6906, 5298, 1388]","[4, 0, 0, 4, 3, 0, 0, 0, 0, 5, 0, 1, 0, 0, 0, ...","[89900, 163127, 109865, 90632, 167568, 99923, ...",9,...,9,These are addicting! I also followed the advic...,chocolate toffee graham treats,25,"['30-minutes-or-less', 'time-to-make', 'course...","[177.7, 19.0, 57.0, 3.0, 1.0, 34.0, 5.0]",12,"['preheat oven to 350', 'arrange graham cracke...",these are delicious and easy (only 5 ingredien...,5
3,560491,167273,5.0,31341,"[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",0,"[840, 6906, 1680, 6389, 1388, 4493]","[497, 10, 3, 169, 257, 0, 1, 50, 11, 456, 21, ...","[12790, 65277, 78451, 104846, 71019, 122679, 5...",1253,...,1253,These were really good! My DSs were a little ...,chocolate chip biscuits aussie style,30,"['30-minutes-or-less', 'time-to-make', 'course...","[98.9, 8.0, 27.0, 1.0, 2.0, 16.0, 4.0]",8,"['cream together the butter and sugar', 'add t...","these are a very ""moresome"" bikky:) i used to ...",6
4,55756,45755,4.0,17323,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, ...",1,"[5574, 340, 6531, 6335, 7946, 7041, 6696, 2809...","[1, 0, 0, 1, 3, 0, 0, 0, 0, 8, 7, 2, 0, 0, 1, ...","[126928, 1108, 40888, 46727, 149767, 117646, 1...",22,...,22,this was very good! i couldn't readily find ho...,szechuan pork crock pot,255,"['weeknight', 'time-to-make', 'course', 'main-...","[416.4, 25.0, 39.0, 52.0, 82.0, 24.0, 7.0]",5,['trim chops of excess fat and brown on both s...,all the pleasure of a chinese meal right out o...,13


----------------------------------------
Unique values of user_id :
[  89366  179133 1812036 ...  716888 1688413  377077]
----------------------------------------
Unique values of recipe_id :
[151621 275036  32871 ... 379967 181797 411112]
----------------------------------------
Unique values of rating :
[5. 4. 0. 3. 2. 1.]
----------------------------------------
Unique values of i :
[165827 104393  99923 ... 156917 138541 125582]
----------------------------------------
Unique values of techniques_recipes :
['[0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0]'
 '[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]'
 '[1, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0,

In [10]:
full_df_processed=DataFramePreprocessing(full_df)
full_df_processed=full_df_processed.preprocessing()

In [18]:
print(f"Columns of the full processed dataframe:")
print(full_df_processed.columns)
print("----------------------------------------")
print(f"Shape of the full processed dataframe:")
print(full_df_processed.shape)
print("----------------------------------------")
print(f"Head of the full  processed dataframe:")
display(full_df_processed.head())
print("----------------------------------------")
for col in full_df_processed.columns:
    print(f"Exemple of {col} :")
    print(full_df_processed.iloc[0][col])
    print("----------------------------------------")
for col in full_df_processed.columns:
    print(f"The type of {col} is")
    print(full_df_processed[col].apply(type).unique())
    print("---------------------------------")

Columns of the full processed dataframe:
Index(['user_id', 'recipe_id', 'rating', 'i', 'techniques_recipes',
       'calorie_level', 'ingredient_ids', 'techniques_users', 'items',
       'n_items', 'ratings', 'n_ratings', 'review', 'name', 'minutes', 'tags',
       'nutrition', 'n_steps', 'steps', 'description', 'n_ingredients'],
      dtype='object')
----------------------------------------
Shape of the full processed dataframe:
(50000, 21)
----------------------------------------
Head of the full  processed dataframe:


,user_id,recipe_id,rating,i,techniques_recipes,calorie_level,ingredient_ids,techniques_users,items,n_items,...,n_ratings,review,name,minutes,tags,nutrition,n_steps,steps,description,n_ingredients
0,89366,151621,5.0,165828,"[0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, ...",2,"[5890, 1954, 5314, 840, 6335, 3203, 7705, 3810...","[4, 1, 0, 1, 2, 0, 1, 0, 0, 4, 0, 0, 0, 1, 0, ...","[109573, 68779, 176014, 14451, 165827, 69725, ...",8,...,8,Loved this! I made sure each of my shrimp were...,shrimp and pasta with creole cream sauce,48,60 minutes or less [SEP] time to make [SEP] co...,"[906.9, 72.0, 2.0, 23.0, 96.0, 135.0, 24.0]",18,peel and devein shrimp [SEP] toss shrimp with ...,from southern living.,10
1,179133,275036,5.0,104394,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",1,"[2991, 3440, 5010, 3203, 840, 5006, 6273, 5180]","[357, 7, 3, 144, 158, 0, 3, 23, 2, 322, 13, 29...","[90890, 135961, 12856, 128256, 37047, 153154, ...",970,...,970,This was very nice! I used red bell pepper and...,easy weeknight corn,15,15 minutes or less [SEP] time to make [SEP] co...,"[270.4, 20.0, 4.0, 3.0, 11.0, 37.0, 12.0]",6,"in pan / dutch oven , add butter , olive oil ,...",i threw some things together in a dutch oven a...,8
2,1812036,32871,5.0,99924,"[1, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, ...",0,"[3729, 840, 6906, 5298, 1388]","[4, 0, 0, 4, 3, 0, 0, 0, 0, 5, 0, 1, 0, 0, 0, ...","[89900, 163127, 109865, 90632, 167568, 99923, ...",9,...,9,These are addicting! I also followed the advic...,chocolate toffee graham treats,25,30 minutes or less [SEP] time to make [SEP] co...,"[177.7, 19.0, 57.0, 3.0, 1.0, 34.0, 5.0]",12,preheat oven to 350 [SEP] arrange graham crack...,these are delicious and easy (only 5 ingredien...,5
3,560491,167273,5.0,31342,"[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",0,"[840, 6906, 1680, 6389, 1388, 4493]","[497, 10, 3, 169, 257, 0, 1, 50, 11, 456, 21, ...","[12790, 65277, 78451, 104846, 71019, 122679, 5...",1253,...,1253,These were really good! My DSs were a little ...,chocolate chip biscuits aussie style,30,30 minutes or less [SEP] time to make [SEP] co...,"[98.9, 8.0, 27.0, 1.0, 2.0, 16.0, 4.0]",8,cream together the butter and sugar [SEP] add ...,"these are a very ""moresome"" bikky:) i used to ...",6
4,55756,45755,4.0,17324,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, ...",1,"[5574, 340, 6531, 6335, 7946, 7041, 6696, 2809...","[1, 0, 0, 1, 3, 0, 0, 0, 0, 8, 7, 2, 0, 0, 1, ...","[126928, 1108, 40888, 46727, 149767, 117646, 1...",22,...,22,this was very good! i couldn't readily find ho...,szechuan pork crock pot,255,weeknight [SEP] time to make [SEP] course [SEP...,"[416.4, 25.0, 39.0, 52.0, 82.0, 24.0, 7.0]",5,trim chops of excess fat and brown on both sid...,all the pleasure of a chinese meal right out o...,13


----------------------------------------
Exemple of user_id :
89366
----------------------------------------
Exemple of recipe_id :
151621
----------------------------------------
Exemple of rating :
5.0
----------------------------------------
Exemple of i :
165828
----------------------------------------
Exemple of techniques_recipes :
[0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0]
----------------------------------------
Exemple of calorie_level :
2
----------------------------------------
Exemple of ingredient_ids :
[5890, 1954, 5314, 840, 6335, 3203, 7705, 3810, 2856, 5180]
----------------------------------------
Exemple of techniques_users :
[4, 1, 0, 1, 2, 0, 1, 0, 0, 4, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 2, 0, 0, 0, 0, 2, 0, 0, 1, 0, 1, 0, 0, 0, 1, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 2, 0, 0]
----------------------------------------

In [13]:
tokenizer=AutoTokenizer.from_pretrained("distilbert-base-uncased")

In [14]:
text_tokenized=tokenizer(full_df_processed.iloc[0]["steps"],return_tensors="pt",max_length=256,padding="max_length",truncation=True)

In [25]:
for key in list(text_tokenized.keys()):
    print(key+"_steps")
    print(type(key))

input_ids_steps
<class 'str'>
attention_mask_steps
<class 'str'>
